# 📈 Linear Regression: Own-Price Elasticity
---

## Econometric Specification

$$\ln(Q_{it}) = \alpha_i + \beta_i \ln(P_{it}) + \gamma'_i \mathbf{X}_{it} + \varepsilon_{it}$$

Where:
- $\beta_i$ = **price elasticity** (our target parameter)
- $\mathbf{X}_{it}$ = control variables (CPI, seasonality)
- $\varepsilon_{it} \sim N(0, \sigma^2)$ (error term)

## Key Property

In log-log form, $\beta = \frac{\partial \ln Q}{\partial \ln P} = \frac{\%\Delta Q}{\%\Delta P} = \varepsilon$

**The coefficient IS the elasticity.**

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import het_breuschpagan
import plotly.express as px
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session
session = get_active_session()

---
## 1. Data Loading

In [ ]:
df = session.sql('SELECT * FROM CHEMICALS_DB.CHEMICAL_OPS.ML_TRAINING_DATA WHERE LN_QUANTITY IS NOT NULL').to_pandas()
print(f'📊 Loaded {len(df):,} observations')

---
## 2. OLS Estimation Function

In [ ]:
def estimate_elasticity(data, product_id, min_obs=50):
    """
    Estimate own-price elasticity for a single product.
    
    Model: ln(Q) = α + β·ln(P) + γ·CPI_std + Σλ_m·Month_m + ε
    
    Returns dict with elasticity, standard error, diagnostics.
    """
    pdata = data[data['PRODUCT_ID'] == product_id].copy()
    if len(pdata) < min_obs:
        return None
    
    y = pdata['LN_QUANTITY']
    X = pd.DataFrame({'const': 1, 'ln_price': pdata['LN_PRICE']})
    
    # Add CPI control (standardized)
    if pdata['FUEL_OIL_CPI'].notna().mean() > 0.5:
        cpi = pdata['FUEL_OIL_CPI'].fillna(pdata['FUEL_OIL_CPI'].mean())
        X['cpi_std'] = (cpi - cpi.mean()) / cpi.std()
    
    # Add month dummies
    dummies = pd.get_dummies(pdata['MONTH'], prefix='m', drop_first=True)
    X = pd.concat([X.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    
    # Clean and fit
    mask = ~(X.isna().any(axis=1) | y.reset_index(drop=True).isna())
    X_clean, y_clean = X[mask.values], y.reset_index(drop=True)[mask.values]
    
    if len(y_clean) < min_obs:
        return None
    
    model = OLS(y_clean, X_clean).fit()
    bp_stat, bp_pval, _, _ = het_breuschpagan(model.resid, X_clean)
    
    return {
        'product_id': product_id,
        'product_name': pdata['PRODUCT_NAME'].iloc[0],
        'family': pdata['PRODUCT_FAMILY'].iloc[0],
        'elasticity': model.params['ln_price'],
        'std_error': model.bse['ln_price'],
        't_stat': model.tvalues['ln_price'],
        'p_value': model.pvalues['ln_price'],
        'ci_lower': model.conf_int().loc['ln_price', 0],
        'ci_upper': model.conf_int().loc['ln_price', 1],
        'r_squared': model.rsquared,
        'n_obs': int(model.nobs),
        'bp_pval': bp_pval
    }

---
## 3. Estimate All Products

In [ ]:
print('🔄 ESTIMATING ELASTICITIES')
print('=' * 80)

results = []
for pid in df['PRODUCT_ID'].unique():
    r = estimate_elasticity(df, pid)
    if r:
        results.append(r)
        sig = '***' if r['p_value']<0.01 else '**' if r['p_value']<0.05 else '*' if r['p_value']<0.1 else ''
        print(f"{r['product_name']:30s} β={r['elasticity']:+7.3f} (SE={r['std_error']:.3f}) {sig:3s} R²={r['r_squared']:.3f}")

edf = pd.DataFrame(results)
print(f'\n✅ Estimated {len(edf)} products')

---
## 4. Results Analysis

### 4.1 Elasticity Distribution

In [ ]:
fig = go.Figure()
edf_sorted = edf.sort_values('elasticity')

fig.add_trace(go.Scatter(
    x=edf_sorted['elasticity'], y=edf_sorted['product_name'], mode='markers',
    marker=dict(size=10, color=edf_sorted['family'].astype('category').cat.codes, colorscale='Viridis'),
    error_x=dict(type='data', symmetric=False,
                 array=edf_sorted['ci_upper']-edf_sorted['elasticity'],
                 arrayminus=edf_sorted['elasticity']-edf_sorted['ci_lower'])
))

fig.add_vline(x=-1, line_dash='dash', line_color='white', annotation_text='Unit Elastic')
fig.add_vline(x=0, line_color='red', line_width=2)

fig.update_layout(title='<b>Price Elasticity with 95% Confidence Intervals</b>',
                  xaxis_title='Price Elasticity (β)', template='plotly_dark', height=600)
fig.show()

### 4.2 Model Quality Summary

In [ ]:
print('📊 MODEL QUALITY SUMMARY')
print('=' * 50)
print(f"Negative elasticity (expected): {(edf['elasticity']<0).sum():>3}/{len(edf)}")
print(f"Significant at 5%:              {(edf['p_value']<0.05).sum():>3}/{len(edf)}")
print(f"Pass heteroskedasticity test:   {(edf['bp_pval']>0.05).sum():>3}/{len(edf)}")
print(f"\nElasticity range: [{edf['elasticity'].min():.3f}, {edf['elasticity'].max():.3f}]")
print(f"Mean R²: {edf['r_squared'].mean():.3f}")

---
## ✅ OLS Estimation Complete

**Proceed to:** `03_sur_elasticity` for cross-price effects